<a href="https://colab.research.google.com/github/jacobhroutzong/coding-challenges/blob/main/tensara/1D_Convolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def conv1d(A,B):
  N = len(A)
  K = len(B)
  r =  (K-1) // 2

  C = [0.0] * N # builds an array C

  for i in range(N): # i -> arr N
    total = 0.0 # initializes target accumulator

    for j in range(K): # i -> arr K
      index = i + j - r # calculates radius

      if 0 <= index < N: # checks radius for out of bounds error
        total += A[index] * B[j]
        # accumulates A[index] * B[j]
    C[i] = total
    # adds value in index



In [4]:

import torch
import triton
import triton.language as tl

@triton.jit
def conv1d_kernel(A, B, C, N: tl.constexpr, K: tl.constexpr, BLOCK: tl.constexpr):
    i = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    r = (K - 1) // 2
    total = tl.full((BLOCK,), 0.0, tl.float32)

    for j in range(K):
        index = i + j - r

        a = tl.load(A + index,
                    mask=(i < N) & (index >= 0) & (index < N),
                    other=0.0)

        b = tl.load(B + j)
        total += a * b

    tl.store(C + i, total, mask=i < N)


def solution(A, B):
    N, K = A.numel(), B.numel()
    C = torch.empty_like(A)

    BLOCK = 128
    grid = (triton.cdiv(N, BLOCK),)

    conv1d_kernel[grid](A, B, C, N, K, BLOCK)

    return C


In [5]:

import torch.nn.functional as F

# Original example
A = torch.tensor([3, 6, 9, 12, 15, 18, 21],
                 device="cuda", dtype=torch.float32)
B = torch.tensor([2, 1, 3],
                 device="cuda", dtype=torch.float32)

C = solution(A, B)
expected = F.conv1d(A[None, None, :],
                    B[None, None, :],
                    padding=len(B)//2).flatten()

print("Triton: ", C.tolist())
print("PyTorch:", expected.tolist())

torch.testing.assert_close(C, expected, atol=1e-4, rtol=1e-4)
print("PASS: Results match!")

# Additional tests
for N, K in [(17, 5), (128, 7), (1000, 31), (4096, 63)]:
    A = torch.randn(N, device="cuda")
    B = torch.randn(K, device="cuda")

    C = solution(A, B)
    expected = F.conv1d(
        A[None, None, :],
        B[None, None, :],
        padding=K//2
    ).flatten()

    torch.testing.assert_close(C, expected, atol=1e-3, rtol=1e-3)
    print(f"PASS: N={N}, K={K}")


Triton:  [21.0, 39.0, 57.0, 75.0, 93.0, 111.0, 57.0]
PyTorch: [21.0, 39.0, 57.0, 75.0, 93.0, 111.0, 57.0]
PASS: Results match!
PASS: N=17, K=5
PASS: N=128, K=7
PASS: N=1000, K=31
PASS: N=4096, K=63
